# Efficiency & Multi-Seed Variance — ML-1M / LightGCN

Addresses two reviewer requirements with **one** set of runs (no extra GPU cost):

1. **Efficiency (R1-D, R5-#3):** wall-clock time and peak GPU memory of AIE / GAIE
   (encoder pre-training + fine-tuning) versus full retraining, plus the
   per-request *apply* cost (one corrected forward pass).
2. **Variance (R1-F-III, R5):** mean ± std of Recall@20, NDCG@20, MI-BF, MI-NG
   (and MI-AUC / MI-ACC) over 3 seeds.

Scope is deliberately minimal: one dataset (ML-1M), one backbone (LightGCN),
seeds {1234, 2026, 7}. The attacked backbone and the injected attack set are
**fixed** (seed 1234); the seed varies encoder initialization, simulated deletion
sampling, batch order, and evaluation negatives — i.e. the variance attributable
to the unlearning method itself, which is what the reviewers questioned
(e.g. "2.3460 vs 2.3575 is not interpretable").

**Runtime estimate (Kaggle T4):** backbone ≈20 min (reused if checkpointed),
retrain reference ≈20 min, 6 unlearning pipelines × ≈50 min ≈ 5 h → **≈6 h total**.
Results are written incrementally to `./logs/eff_var_ml1m_lightgcn.json`;
re-running the notebook skips completed entries, so a killed session loses at
most one pipeline.

### Hyperparameters
All values are **inline** in Section 1 — no datasets to attach, no secrets to
configure. They match the effective ml1m/lightgcn configuration of the
committed 3x3 notebooks. (The paper still discloses only symbols and search
ranges; these values were already public in the repo's committed notebooks.)

### Kaggle setup
Settings → Accelerator → **GPU**. That's all.

---
## 0. Environment

In [ ]:
import os
import subprocess
import torch
cuda_tag = torch.version.cuda.replace(".", "")        # e.g. "121" or "124"
torch_tag = ".".join(torch.__version__.split(".")[:2]) # e.g. "2.6"
whl_url = f"https://data.pyg.org/whl/torch-{torch_tag}.0+cu{cuda_tag}.html"
print(f"Installing torch-scatter + torch-sparse from: {whl_url}")
subprocess.check_call(["pip", "install", "-q", "torch-scatter", "torch-sparse", "-f", whl_url])
subprocess.check_call(["pip", "install", "-q", "setproctitle"])

import torch_scatter
import torch_sparse
import setproctitle
print("torch_scatter version:", torch_scatter.__version__)
print("torch_sparse version:", torch_sparse.__version__)
print("setproctitle installed \u2713")

In [ ]:
import os

REPO_URL = "https://github.com/Shuvayu12/unlearnrec_improv.git"
PROJECT_DIR = "/kaggle/working/unlearnrec_improv"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already cloned.")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
import sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Clear sys.argv so argparse in config/params.py doesn't choke on notebook kernel args
sys.argv = [sys.argv[0]]

from config.params import args
from data.data_handler import DataHandler
from Utils.time_logger import log
from models.Model import LightGCN, AIE, GAIE
from training.pretrain_simgcl import Coach as PretrainCoach   # dispatches on args.model
from unlearning.aie_unlearn import Coach as AIECoach
from unlearning.gaie_unlearn import Coach as GAIECoach

import torch as t
import numpy as np
import random
import time
import json
import glob
import traceback

os.makedirs("./ckpt", exist_ok=True)
os.makedirs("./logs", exist_ok=True)

print(f"CUDA available: {t.cuda.is_available()}")
if t.cuda.is_available():
    print(f"GPU: {t.cuda.get_device_name(0)}")

---
## 1. Hyperparameters (inline — zero setup)

The values below are the effective ml1m/lightgcn configuration of the
committed 3x3 notebooks: `UNLEARN_CFG` plus the `STRONG_UNLEARN` override that
applies to every LightGCN combo, and the per-pipeline fine-tune settings.

Key ↔ paper-symbol map: `epoch`→N_pre, `pretrain_drop_rate`→ρ_s,
`sim_epoch`→T_sim, `unlearn_wei`→λ_u, `align_wei`→λ_p, `align_temp`→τ,
`rec_wei`→λ_r (GAIE), fine-tune `epoch`→N_ft, `unlearn_wei`→λ'_u,
`align_wei`→λ'_p, `reg`→γ.

**Optional override (not required):** if an `hparams.json` is supplied via the
`HPARAMS_PATH` env var or any attached Kaggle input dataset, its non-null keys
replace the inline defaults — handy for trying other values without editing
the notebook. When nothing is attached, the defaults just run.

In [ ]:
# Effective ml1m/lightgcn configuration, identical to the committed 3x3
# notebooks (UNLEARN_CFG + STRONG_UNLEARN override for LightGCN combos).
HP = {
    'unlearn':       dict(epoch=30, batch=4096, pretrain_drop_rate=0.1,
                          sim_epoch=10, unlearn_wei=1.0, align_wei=0.01,
                          align_temp=1.0, perf_degrade=0.3),
    'gaie_extra':    dict(rec_wei=0.03, perf_degrade=0.5),
    'finetune_aie':  dict(epoch=20, unlearn_wei=0.2, align_wei=0.01),
    'finetune_gaie': dict(epoch=15, unlearn_wei=0.2, align_wei=0.01),
    'backbone':      dict(pretrain_epoch=50, batch=2048, reg=1e-7,
                          adv_method='lightgcn0.5'),
}
HP_SOURCE = 'inline notebook defaults (ml1m/lightgcn, matches the 3x3 notebooks)'

# Optional override -- no setup required. If an hparams.json exists at
# HPARAMS_PATH or inside an attached Kaggle input dataset, its non-null
# values replace the defaults.
_cand = os.environ.get('HPARAMS_PATH')
if not (_cand and os.path.exists(_cand)):
    _hits = sorted(glob.glob('/kaggle/input/*/hparams.json'))
    _cand = _hits[0] if _hits else None
if _cand:
    with open(_cand) as fs:
        _ext = json.load(fs)
    for _sec, _vals in _ext.items():
        HP.setdefault(_sec, {}).update(
            {k: v for k, v in _vals.items() if v is not None})
    HP_SOURCE = f'inline defaults overridden by {_cand}'

print('Hyperparameters:', HP_SOURCE)

---
## 2. Configuration & harness

In [ ]:
DATASET = 'ml1m'
BACKBONE = 'lightgcn'
SEEDS = [1234, 2026, 7]
PIPELINES = ['aie', 'gaie']
FT_TUNE_E0 = True          # paper protocol: E0 unfrozen during fine-tuning
BACKBONE_SEED = 1234       # attacked backbone + attack set fixed across seeds

RESULTS_FILE = './logs/eff_var_ml1m_lightgcn.json'
all_results = {}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as fs:
        all_results = json.load(fs).get('results', {})
    print(f'Resuming: {len(all_results)} entries already in {RESULTS_FILE}')


def save_results():
    with open(RESULTS_FILE, 'w') as fs:
        json.dump({'dataset': DATASET, 'backbone': BACKBONE, 'seeds': SEEDS,
                   'ft_tune_e0': FT_TUNE_E0,
                   'protocol': 'attacked-backbone (UnlearnRec Sec 4.1.4)',
                   'results': all_results}, fs, indent=2)


def reset_seeds(seed):
    t.manual_seed(seed)
    t.cuda.manual_seed_all(seed)
    t.backends.cudnn.deterministic = True
    np.random.seed(seed)
    random.seed(seed)


def apply_args(d):
    for k, v in d.items():
        setattr(args, k, v)


def set_common(seed):
    args.data = DATASET
    args.model = BACKBONE
    args.seed = seed            # DataHandler re-seeds from args.seed
    args.gpu = '0'
    args.lr = 1e-3
    args.latdim = 128
    args.gnn_layer = 3
    args.topk = 20
    args.tst_bat = 256
    args.decay = 1.0
    args.bpr_wei = 1.0
    args.load_model = None
    args.reg = HP['backbone']['reg']
    args.adv_method = HP['backbone']['adv_method']
    args.adversarial_attack = True


# Architecture / loss-variant settings disclosed in the paper.
UNLEARN_STRUCTURAL = dict(
    lr=1e-3, test_drop_rate=0.003, tst_epoch=5, bpr_wei=1.0,
    align_type='v2', unlearn_type='v1', overall_withdraw_rate=0.1,
    withdraw_rate_init=1, hyper_temp=1.0, unlearn_ssl=1e-3,
    layer_mlp=2, leaky=0.99, act='leaky', unlearn_layer=0,
)


def unlearn_cfg(pipeline):
    cfg = dict(UNLEARN_STRUCTURAL)
    u = HP['unlearn']
    cfg.update(epoch=u['epoch'], batch=u['batch'],
               pretrain_drop_rate=u['pretrain_drop_rate'], sim_epoch=u['sim_epoch'],
               unlearn_wei=u['unlearn_wei'], align_wei=u['align_wei'],
               align_temp=u['align_temp'], perf_degrade=u['perf_degrade'])
    if pipeline == 'gaie':
        g = HP['gaie_extra']
        cfg['rec_wei'] = g['rec_wei']
        cfg['perf_degrade'] = g.get('perf_degrade', cfg['perf_degrade'])
    return cfg


def timed_stage(fn):
    """Run fn() and return (result, wall_seconds, peak_gpu_GiB)."""
    t.cuda.synchronize()
    t.cuda.reset_peak_memory_stats()
    t0 = time.time()
    out = fn()
    t.cuda.synchronize()
    return out, time.time() - t0, t.cuda.max_memory_allocated() / 2**30


print(f'Config: {DATASET}/{BACKBONE}, seeds={SEEDS}, pipelines={PIPELINES}')

---
## 3. Attacked backbone (fixed, seed 1234)

The model to be unlearned: LightGCN trained on clean + injected adversarial
edges. Trained once and reused across all seeds, so every pipeline unlearns the
**same** model and the **same** request — the seed then isolates the unlearning
method's own variance.

In [ ]:
set_common(BACKBONE_SEED)
args.epoch = HP['backbone']['pretrain_epoch']
args.batch = HP['backbone']['batch']
args.tst_epoch = 10
BACKBONE_CKPT = f'./ckpt/pre_{DATASET}_{BACKBONE}_adv'
args.save_path = BACKBONE_CKPT
reset_seeds(BACKBONE_SEED)

handler_atk = DataHandler()
handler_atk.load_data(drop_rate=0.0, adv_attack=True)
print(f'users={args.user}, items={args.item}, '
      f'injected adversarial edges={len(handler_atk.adv_edges[0])}')

if not os.path.exists(BACKBONE_CKPT + '.mod'):
    _t0 = time.time()
    PretrainCoach(handler_atk).run()
    print(f'Attacked backbone trained in {time.time() - _t0:.1f}s')
else:
    print(f'Reusing checkpoint {BACKBONE_CKPT}.mod')

_m = t.load(BACKBONE_CKPT + '.mod', weights_only=False)['model'].cuda()
_m.training = False
_res = PretrainCoach(handler_atk).tst_epoch(_m)
print(f'[Attacked backbone] Recall@{args.topk}={_res["Recall"]:.4f}, '
      f'NDCG@{args.topk}={_res["NDCG"]:.4f}')
BACKBONE_RES = dict(_res)
del _m, handler_atk
t.cuda.empty_cache()

---
## 4. Retrain reference — timed (the efficiency baseline)

Exact unlearning: the same backbone retrained from scratch on the clean
(residual) graph, with wall-clock and peak GPU memory recorded. Timing is
persisted to the results JSON so a resumed session doesn't need to retrain
just to recover the number.

In [ ]:
if 'retrain_ref' in all_results:
    print('Reusing persisted retrain timing:',
          {k: round(v, 2) for k, v in all_results['retrain_ref'].items()})
else:
    set_common(BACKBONE_SEED)
    args.adversarial_attack = False
    args.epoch = HP['backbone']['pretrain_epoch']
    args.batch = HP['backbone']['batch']
    args.tst_epoch = 10
    args.save_path = './ckpt/retrain_ref'
    reset_seeds(BACKBONE_SEED)

    handler_clean = DataHandler()
    handler_clean.load_data(drop_rate=0.0, adv_attack=False)

    _, t_retrain, mem_retrain = timed_stage(PretrainCoach(handler_clean).run)

    _m = t.load('./ckpt/retrain_ref.mod', weights_only=False)['model'].cuda()
    _m.training = False
    _res = PretrainCoach(handler_clean).tst_epoch(_m)

    all_results['retrain_ref'] = dict(
        Time=t_retrain, PeakMemGiB=mem_retrain,
        Recall=_res['Recall'], NDCG=_res['NDCG'])
    save_results()
    print(f'Retrain reference: {t_retrain:.1f}s, peak {mem_retrain:.2f} GiB, '
          f'Recall={_res["Recall"]:.4f}, NDCG={_res["NDCG"]:.4f}')
    del _m, handler_clean
    t.cuda.empty_cache()

---
## 5. Per-seed unlearning runs (AIE + GAIE × 3 seeds)

Each run: encoder pre-training on simulated deletions → fine-tuning (E0
unfrozen) → evaluation of the best checkpoint on the actual request.
Per stage we record wall-clock and peak GPU memory; at the end, the cost of
*applying* the unlearning (one corrected forward pass) — the per-request
inference cost once the encoder exists.

Note: final evaluation calls `model.eval()`, so GAIE uses Z = μ
deterministically as stated in the paper (Sec. 3.2).

In [ ]:
COACHES = {'aie': AIECoach, 'gaie': GAIECoach}


def run_seed(pipeline, seed):
    coach_cls = COACHES[pipeline]

    # ---- stage 1: encoder pre-training on simulated deletions ----
    set_common(seed)
    apply_args(unlearn_cfg(pipeline))
    args.fineTune = False
    args.trained_model = BACKBONE_CKPT
    args.save_path = f'./ckpt/{pipeline}_{DATASET}_{BACKBONE}_s{seed}'
    reset_seeds(seed)
    h_u = DataHandler()
    h_u.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)
    coach = coach_cls(h_u)
    _, t_unlearn, mem_unlearn = timed_stage(coach.run)

    # ---- stage 2: fine-tuning (E0 unfrozen) ----
    args.fineTune = True
    args.ft_tune_e0 = FT_TUNE_E0
    args.model_2_finetune = args.save_path
    ft = HP[f'finetune_{pipeline}']
    apply_args(dict(epoch=ft['epoch'], unlearn_wei=ft['unlearn_wei'],
                    align_wei=ft['align_wei']))
    args.save_path = args.save_path + '_ft'
    reset_seeds(seed)
    h_f = DataHandler()
    h_f.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)
    coach = coach_cls(h_f)
    _, t_ft, mem_ft = timed_stage(coach.run)

    # ---- evaluation: best checkpoint, actual request, deterministic ----
    h_e = DataHandler()
    h_e.load_data(drop_rate=args.test_drop_rate, adv_attack=True)
    sp = args.save_path + '.mod'
    if os.path.exists(sp):
        coach.model = t.load(sp, weights_only=False)['model'].cuda()
    coach.handler = h_e
    coach.model.eval()      # GAIE: Z = mu at inference (paper Sec 3.2)
    reset_seeds(seed)       # reproducible evaluation negatives
    metrics = coach.tst_epoch(coach.model)
    mi = coach.test_unlearn(coach.model, prefix=f'[{pipeline.upper()} seed {seed}] final')

    # ---- per-request apply cost: one corrected forward pass ----
    with t.no_grad():       # warm-up, then timed
        coach.model.outforward(h_e.ts_ori_adj, h_e.ts_pk_adj, h_e.mask, h_e.ts_drp_adj)
    t.cuda.synchronize()
    _t0 = time.time()
    with t.no_grad():
        coach.model.outforward(h_e.ts_ori_adj, h_e.ts_pk_adj, h_e.mask, h_e.ts_drp_adj)
    t.cuda.synchronize()
    t_apply_ms = (time.time() - _t0) * 1000

    row = dict(seed=seed,
               Recall=metrics['Recall'], NDCG=metrics['NDCG'],
               MI_BF=mi['mi_bf'], MI_NG=mi['mi_ng'],
               MI_AUC=mi['mi_auc'], MI_ACC=mi['mi_acc'],
               BeforeProb=mi['avg_before_prob'], AfterProb=mi['avg_after_prob'],
               NegProb=mi['avg_neg_prob'],
               UnlearnTime=t_unlearn, FinetuneTime=t_ft,
               UnlearnPeakMemGiB=mem_unlearn, FinetunePeakMemGiB=mem_ft,
               ApplyMs=t_apply_ms,
               BackboneRecall=BACKBONE_RES['Recall'], BackboneNDCG=BACKBONE_RES['NDCG'])
    del coach, h_u, h_f, h_e
    t.cuda.empty_cache()
    return row

In [ ]:
for seed in SEEDS:
    for pipeline in PIPELINES:
        key = f'{pipeline}/seed{seed}'
        if key in all_results and 'error' not in all_results[key]:
            print(f'skip {key} (already done)')
            continue
        print('\n' + '#' * 100)
        print(f'### {pipeline.upper()} — {DATASET}/{BACKBONE} — seed {seed}')
        print('#' * 100)
        try:
            all_results[key] = run_seed(pipeline, seed)
        except Exception:
            traceback.print_exc()
            all_results[key] = {'error': traceback.format_exc()[-1500:]}
        save_results()
        print(f'saved {RESULTS_FILE}')

---
## 6. Variance report (mean ± std over seeds)

In [ ]:
METRICS = ['Recall', 'NDCG', 'MI_BF', 'MI_NG', 'MI_AUC', 'MI_ACC']


def rows_for(pipeline):
    return [all_results[f'{pipeline}/seed{s}'] for s in SEEDS
            if f'{pipeline}/seed{s}' in all_results
            and 'error' not in all_results[f'{pipeline}/seed{s}']]


stats = {}
for p in PIPELINES:
    rr = rows_for(p)
    if not rr:
        continue
    stats[p] = {m: (float(np.mean([r[m] for r in rr])),
                    float(np.std([r[m] for r in rr], ddof=1)) if len(rr) > 1 else 0.0,
                    len(rr))
                for m in METRICS}

print(f'{DATASET}/{BACKBONE} — mean ± std over {len(SEEDS)} seeds {SEEDS}\n')
hdr = 'method  ' + ''.join(f'{m:>20}' for m in METRICS)
print(hdr)
print('-' * len(hdr))
for p, st in stats.items():
    cells = ''.join(f"{st[m][0]:>12.4f} ±{st[m][1]:<6.4f}" for m in METRICS)
    print(f'{p.upper():<8}{cells}   (n={st[METRICS[0]][2]})')

print('\nper-seed rows:')
for p in PIPELINES:
    for r in rows_for(p):
        print(f"  {p:<5} seed {r['seed']:<5} " +
              ' '.join(f'{m}={r[m]:.4f}' for m in METRICS))

print('\n% ---- LaTeX (variance table) ----')
print(r'\begin{tabular}{l' + 'c' * len(METRICS) + '}')
print(r'\toprule')
print('Method & ' + ' & '.join(m.replace('_', '-') for m in METRICS) + r' \\')
print(r'\midrule')
for p, st in stats.items():
    cells = ' & '.join(f'{st[m][0]:.4f} $\\pm$ {st[m][1]:.4f}' for m in METRICS)
    print(f'{p.upper()} & {cells} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')

---
## 7. Efficiency report

Two framings, both of which the paper should state (reviewer R1-D's point about
new requests needing another optimisation round):

- **Total cost** = encoder pre-training + fine-tuning, i.e. the worst case where
  everything happens after the request arrives.
- **Per-request cost** = fine-tuning only: the encoder is pre-trained *once per
  deployed model* on simulated deletions (before any request exists), so only
  the fine-tune stage is on the request's critical path. **Apply (ms)** is the
  cost of serving the corrected embeddings once the encoder is trained.

In [ ]:
ret = all_results.get('retrain_ref')

print(f'{"method":<10} {"pretrain(s)":>14} {"finetune(s)":>14} {"total(s)":>10} '
      f'{"speedup":>8} {"ft-only":>8} {"apply(ms)":>10} {"peak GiB":>9}')
print('-' * 92)
if ret:
    print(f'{"Retrain":<10} {"-":>14} {"-":>14} {ret["Time"]:>10.1f} '
          f'{"1.0x":>8} {"1.0x":>8} {"-":>10} {ret["PeakMemGiB"]:>9.2f}')
else:
    print('Retrain    N/A — run Section 4 first')

eff = {}
for p in PIPELINES:
    rr = rows_for(p)
    if not rr:
        continue
    mu = lambda k: float(np.mean([r[k] for r in rr]))
    sd = lambda k: float(np.std([r[k] for r in rr], ddof=1)) if len(rr) > 1 else 0.0
    tot = mu('UnlearnTime') + mu('FinetuneTime')
    eff[p] = dict(pre=mu('UnlearnTime'), pre_sd=sd('UnlearnTime'),
                  ft=mu('FinetuneTime'), ft_sd=sd('FinetuneTime'), tot=tot,
                  mem=max(mu('UnlearnPeakMemGiB'), mu('FinetunePeakMemGiB')),
                  apply_ms=mu('ApplyMs'))
    sp_tot = f"{ret['Time'] / tot:.1f}x" if ret else 'N/A'
    sp_ft = f"{ret['Time'] / mu('FinetuneTime'):.1f}x" if ret else 'N/A'
    print(f'{p.upper():<10} {mu("UnlearnTime"):>7.1f} ±{sd("UnlearnTime"):<5.1f} '
          f'{mu("FinetuneTime"):>7.1f} ±{sd("FinetuneTime"):<5.1f} {tot:>10.1f} '
          f'{sp_tot:>8} {sp_ft:>8} {mu("ApplyMs"):>10.1f} {eff[p]["mem"]:>9.2f}')

print('\n% ---- LaTeX (efficiency table) ----')
print(r'\begin{tabular}{lccccc}')
print(r'\toprule')
print(r'Method & Pre-train (s) & Fine-tune (s) & Total (s) & Speedup & Peak mem (GiB) \\')
print(r'\midrule')
if ret:
    print(f'Retrain & -- & -- & {ret["Time"]:.0f} & 1.0$\\times$ & {ret["PeakMemGiB"]:.2f} \\\\')
for p, e in eff.items():
    sp = f'{ret["Time"] / e["tot"]:.1f}' if ret else '--'
    print(f'{p.upper()} & {e["pre"]:.0f} $\\pm$ {e["pre_sd"]:.0f} & '
          f'{e["ft"]:.0f} $\\pm$ {e["ft_sd"]:.0f} & {e["tot"]:.0f} & '
          f'{sp}$\\times$ & {e["mem"]:.2f} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')

---
## Notes for the paper

- **What varies with the seed:** encoder initialization, simulated-deletion
  sampling, mini-batch order, BPR negatives, and evaluation negatives. The
  attacked backbone, attack set, and data split are fixed — so the ± reflects
  the unlearning method, not backbone training noise. Say this in the caption.
- **MI-AUC / MI-ACC** are genuine membership-attack numbers (threshold sweep
  over deleted-vs-negative scores) — reporting them answers the "MI-BF/MI-NG
  are only probability ratios" criticism directly.
- **Efficiency claims:** quote the total-cost speedup as the conservative
  number, and the fine-tune-only speedup as the amortized per-request cost
  (encoder pre-training happens once per model, before any request arrives).
- Retrain reference uses the same epoch budget as backbone training; if the
  full-run logs show convergence earlier, the speedup is understated (safe
  direction).